In [42]:
import numpy as np
import os
import kaggle
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv3D,MaxPooling3D,Flatten,Dense,Dropout,BatchNormalization
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

In [43]:
train=pd.read_csv('/kaggle/input/competitions/3-d-cn-ns-on-geometric-shapes/train/train.csv')
print(len(train))

x=[]
y=[]
for _, row in train.iterrows():
    voxel_path=os.path.join('/kaggle/input/competitions/3-d-cn-ns-on-geometric-shapes','train_voxels',row['id']+'.npy')
    voxel=np.load(voxel_path)
    x.append(voxel)
    y.append(row['label'])
X=np.array(x)
Y=np.array(y)


60000


In [44]:
encoder=LabelEncoder()
Y=encoder.fit_transform(Y)

In [45]:
X=X[...,np.newaxis]

In [75]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import  GlobalAveragePooling3D
from tensorflow.keras.regularizers import l2
model = Sequential()
model.add(Conv3D(32, (3,3,3), padding='same', activation='relu', input_shape=(20,20,20,1)))
model.add(BatchNormalization())
model.add(Conv3D(32, (3,3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling3D((2,2,2)))
model.add(Dropout(0.1))

model.add(Conv3D(64, (3,3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv3D(64, (3,3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling3D((2,2,2)))
model.add(Dropout(0.2))

model.add(Conv3D(128, (3,3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(MaxPooling3D((2,2,2)))
model.add(Dropout(0.2))

model.add(GlobalAveragePooling3D())
model.add(Dense(128, activation='relu', kernel_regularizer=l2(1e-5)))
model.add(Dropout(0.4))
model.add(Dense(7, activation='softmax'))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [76]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [77]:
X_train,X_val,y_train,y_val=train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=0,
    stratify=Y
                
)

In [78]:
def augment(voxel, label):
    if tf.random.uniform([]) > 0.5:
        voxel = tf.reverse(voxel, axis=[0])
    if tf.random.uniform([]) > 0.5:
        voxel = tf.reverse(voxel, axis=[1])
    if tf.random.uniform([]) > 0.5:
        voxel = tf.reverse(voxel, axis=[2])
    return voxel, label

In [79]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = (
    train_ds
    .shuffle(buffer_size=2048)
    .map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((X_val, y_val))
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

In [80]:
model.fit(
    train_ds,                      
    epochs=50,
    validation_data=val_ds,        
    callbacks=callbacks
)


Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 81s 46ms/step - accuracy: 0.4220 - loss: 1.2897 - val_accuracy: 0.5811 - val_loss: 0.8797 - learning_rate: 0.0015
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 68s 45ms/step - accuracy: 0.5940 - loss: 0.8162 - val_accuracy: 0.5799 - val_loss: 0.8345 - learning_rate: 0.0015
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 68s 45ms/step - accuracy: 0.6416 - loss: 0.7173 - val_accuracy: 0.6547 - val_loss: 0.6516 - learning_rate: 0.0015
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 68s 45ms/step - accuracy: 0.6778 - loss: 0.6495 - val_accuracy: 0.7020 - val_loss: 0.5915 - learning_rate: 0.0015
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 68s 45ms/step - accuracy: 0.7176 - loss: 0.5653 - val_accuracy: 0.7206 - val_loss: 0.5371 - learning_rate: 7.5000e-04
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 68s 45ms/step - accuracy: 0.7268 - loss: 0.5402 - val_accuracy: 0.7525 - val_loss: 0.4921 - learning_rate: 7.5000e-04
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 68s 45ms/step 

In [84]:
model.save('final_model.keras')
print("Model saved!")

Model saved!


In [85]:
prediction=model.predict(val_ds)
predicted_encodes=np.argmax(prediction,axis=1)
pred=encoder.inverse_transform(predicted_encodes)

375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step


In [86]:
accuracy = accuracy_score(y_val, predicted_encodes)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.81725


In [87]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_val, predicted_encodes)
print(cm)
print(encoder.classes_)

[[1579    3    3   16    1  111    1]
 [   0 1670   44    0    0    0    0]
 [   3  414 1277   15    0    5    0]
 [  15    8   35 1651    0    5    0]
 [   0    0    0    0 1353    0  361]
 [ 185    6   17    7    0 1499    1]
 [   1    0    0    0  936    0  778]]
['cone' 'cube' 'cuboid' 'cylinder' 'ellipsoid' 'pyramid' 'sphere']


In [92]:

import numpy as np
import pandas as pd
import os
test = pd.read_csv(
    '/kaggle/input/competitions/3-d-cn-ns-on-geometric-shapes/test/test.csv'
)

print("Number of test samples:", len(test))
print(test.head())
X_test = []

for _, row in test.iterrows():

    voxel_path = os.path.join(
        '/kaggle/input/competitions/3-d-cn-ns-on-geometric-shapes',
        'test_voxels',
        row['id'] + '.npy'
    )
    voxel = np.load(voxel_path)
    X_test.append(voxel)


X_test = np.array(X_test)

print("Test voxel shape:", X_test.shape)
X_test = X_test[..., np.newaxis]

print("Final test shape:", X_test.shape)



prediction = model.predict(
    X_test,
    batch_size=32
)



predicted_encodes = np.argmax(
    prediction,
    axis=1
)



predicted_labels = encoder.inverse_transform(
    predicted_encodes
)



submission = pd.DataFrame({
    'id': test['id'],
    'label': predicted_labels
})



submission.to_csv(
    '/kaggle/working/submission.csv',
    index=False
)


Number of test samples: 10000
            id
0  test_000000
1  test_000001
2  test_000002
3  test_000003
4  test_000004
Test voxel shape: (10000, 20, 20, 20)
Final test shape: (10000, 20, 20, 20, 1)
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step


In [93]:
print('hello')

hello


In [94]:
import os

print("Exists:", os.path.exists('/kaggle/working/submission.csv'))

if os.path.exists('/kaggle/working/submission.csv'):
    print("Size:", os.path.getsize('/kaggle/working/submission.csv'), "bytes")
    print(pd.read_csv('/kaggle/working/submission.csv').shape)

Exists: True
Size: 193470 bytes
(10000, 2)
